# 07.02_Auco_seurat_analysis_AucoSpatial_R

Seurat 分支（实际输入为单细胞矩阵）。

- 当前文件：`analysis/07_spatial_analysis/07.02_Auco_seurat_analysis_AucoSpatial_R.ipynb`
- 原始来源：`Codes/07.02_R_seurat_analysis_AucoSpatial.ipynb`（旧编号仅用于溯源）。
- 运行内核：**R**。
- 导入依赖：`DT`, `Seurat`, `dplyr`, `ggplot2`, `htmltools`, `patchwork`。
- 当前编号与流程见 `docs/workflow.md`、`docs/code_index.md`。
- 仅更新整理版导读；原分析单元格、参数和顺序保持不变。原始 cell 索引在本文件中加 1。

**本文件说明：** 文件名含 Spatial，但实际读取 AU_scMatrix_CycloneSeq 单细胞矩阵，并写出其 analysis_results/Auco.seurat.rds；按待核实的单细胞分支阅读。


r_base

# Auco

## 数据处理部分

In [ ]:
library(Seurat)
library(ggplot2)
library(dplyr)
library(DT)
library(htmltools)

### 1. 数据读取与合并

In [ ]:
work_dir = '/share/home/zhangze/zz/NeuralOrigin/Data/03.SingleCellProcessing/scMatrix/AU_scMatrix_CycloneSeq/'

# 读取批次 1 的数据
data_au1 <- Read10X(data.dir = paste0(work_dir,'au.coerulea.QDv2.CycloneSeq.opt.202312batch.01.filter_matrix/'), gene.column = 1)
seurat_au1 <- CreateSeuratObject(counts = data_au1, project = "au1")
seurat_au1$batch <- "1"  # 为批次 1 添加批次信息

# 读取批次 2 的数据
data_au2 <- Read10X(data.dir = paste0(work_dir,'au.coerulea.QDv2.CycloneSeq.opt.202312batch.02.filter_matrix/'), gene.column = 1)
seurat_au2 <- CreateSeuratObject(counts = data_au2, project = "au2")
seurat_au2$batch <- "2"  # 为批次 2 添加批次信息

# 读取批次 3 的数据
data_au3 <- Read10X(data.dir = paste0(work_dir,'au.coerulea.QDv2.CycloneSeq.opt.202312batch.03.filter_matrix/'), gene.column = 1)
seurat_au3 <- CreateSeuratObject(counts = data_au3, project = "au3")
seurat_au3$batch <- "3"  # 为批次 3 添加批次信息

In [ ]:
# 获取两个数据集的基因名
genes_au1 <- rownames(seurat_au1)
genes_au2 <- rownames(seurat_au2)
genes_au3 <- rownames(seurat_au3)
length(genes_au1)
length(genes_au2)
length(genes_au3)

# 获取两个数据集共有的基因
common_genes <- intersect(intersect(genes_au1, genes_au2), genes_au3)

# 打印共有基因的数量
length(common_genes)

# 检查重复基因
anyDuplicated(common_genes)

In [ ]:
handle_object <- merge(seurat_au1, y = c(seurat_au2, seurat_au3), add.cell.ids = c("1", "2", "3"))
handle_object <- subset(handle_object, features = common_genes)

In [ ]:
handle_object
length(rownames(handle_object))

### 2. 线粒体基因比例计算与过滤

In [ ]:
handle_object[["percent.mt"]] = PercentageFeatureSet(handle_object, pattern = "^MT-") # 计算每个细胞的线粒体比例
head(handle_object@meta.data)  # 计算结果保存于细胞的元数据表格中：percent.mt

In [ ]:
p <- VlnPlot(handle_object, features = c("nFeature_RNA", "nCount_RNA", "percent.mt"), ncol = 3)
pdf(paste0(work_dir,'analysis_results/01.VlnPlot.before.nFeature_RNA.qc.pdf')) # 保存图片
plot(p)
dev.off()
p

In [ ]:
p <- hist(handle_object@meta.data$nFeature_RNA, breaks = 200, main = "Distribution Histogram", xlab = "Values", ylab = "Frequency") # 画细胞基因表达量分布图
pdf(paste0(work_dir,'analysis_results/02.before.nFeature_RNA.qc.pdf')) # 保存图片
plot(p)
dev.off()
p

### 3. 质量控制

In [ ]:
sq1 = quantile(handle_object@meta.data$nFeature_RNA, 0.01)  # 计算细胞中基因表达量的的1%分位数，结果保存为sq1
sq2 = quantile(handle_object@meta.data$nFeature_RNA, 0.99)  # 计算细胞中基因表达量的的99%分位数，结果保存为sq2
handle_object = subset(handle_object, subset = nFeature_RNA > sq1 & nFeature_RNA < sq2) # 过滤，保留表达量大于sq1且小于sq2的细胞
p <- hist(handle_object@meta.data$nFeature_RNA,breaks = 200, main = "Distribution Histogram", xlab = "Values", ylab = "Frequency")  # 观察过滤效果
pdf(paste0(work_dir,'analysis_results/03.after.nFeature_RNA.qc.pdf'))
plot(p)
dev.off()
p

In [ ]:
handle_object = NormalizeData(handle_object, normalization.method = "LogNormalize", scale.factor = 5000)
handle_object@assays$RNA@layers$data[1:4,1:4] # 观察正则化之后的矩阵

In [ ]:
handle_object
length(rownames(handle_object))

### 4. 高变基因检测

In [ ]:
handle_object = FindVariableFeatures(handle_object, selection.method = "vst", nfeatures = 2000) # 计算高变基因
top10 = head(VariableFeatures(handle_object), 10)  # 获取top10的高变基因
p1 = VariableFeaturePlot(handle_object)  # 画图展示高变基因的分布
p2 = LabelPoints(plot = p1, points = top10, repel = TRUE)  # 将top10高变基因标在图中
# 保存图片
pdf(paste0(work_dir,'analysis_results/04.VariableFeatures.pdf'))
plot(p2)
dev.off()
p2

In [ ]:
# 数据缩放
handle_object = ScaleData(handle_object, features =  rownames(handle_object))
handle_object@assays$RNA@layers$scale.data[1:4,1:4] # 观察归一化后的矩阵

### 5. PCA 降维

In [ ]:
# PCA 降维
handle_object <- RunPCA(handle_object, features = VariableFeatures(object = handle_object), verbose = FALSE)

In [ ]:
p <- DimPlot(handle_object, reduction = "pca")
# 保存图片
pdf(paste0(work_dir,'analysis_results/05.DimPlot.pca.pdf'))
print(p)
dev.off()
p

In [ ]:
DimHeatmap(handle_object, dims = 1, cells = 500, balanced = TRUE)
DimHeatmap(handle_object, dims = 1:12, cells = 500, balanced = TRUE)

In [ ]:
# 计算
handle_object = JackStraw(handle_object, num.replicate = 100)
handle_object = ScoreJackStraw(handle_object, dims = 1:20)

# 画图
p1 = JackStrawPlot(handle_object, dims = 1:20)
p2 = ElbowPlot(handle_object, ndims = 50)   # 绘制肘部图（选择合适的主成分数量）

# 保存图片
pdf(paste0(work_dir,'analysis_results/06.JackStrawPlot.pdf'))
print(p1)
dev.off()
p1

pdf(paste0(work_dir,'analysis_results/07.ElbowPlot.pdf'))
print(p2)
dev.off()
p2

### 6. 聚类分析

In [ ]:
# step1：构建NNG
handle_object <- FindNeighbors(handle_object, dims = 1:30)
# step2：Louvain algorithm
handle_object <- FindClusters(handle_object, resolution = 0.6)

In [ ]:
# 画图
handle_object <- RunUMAP(handle_object, dims = 1:30)
p <- DimPlot(handle_object, reduction = "umap", label = T)
# 保存图片
pdf(paste0(work_dir,'analysis_results/08.umap.after.cluster.pdf'))
print(p)
dev.off()
p

In [ ]:
# 画图
handle_object <- RunTSNE(handle_object, dims = 1:30)
p <- DimPlot(handle_object, reduction = "tsne", label = T)
# 保存图片
pdf(paste0(work_dir,'analysis_results/09.tsne.after.cluster.pdf'))
print(p)
dev.off()
p

In [ ]:
# 确认当前活跃的分组
# Idents(handle_object)

# 如果不是 `seurat_clusters`，则手动设置
Idents(handle_object) <- "seurat_clusters"
handle_object <- JoinLayers(handle_object)

In [ ]:
handle_object
length(rownames(handle_object))

"RHG32-XENLA#Q6GPD0" %in% rownames(handle_object)

### 7. 差异基因分析

In [ ]:
library(DT)
library(dplyr)

# 你的原始代码 (marker基因鉴定和排序)
markers <- FindAllMarkers(handle_object, only.pos = TRUE, min.pct = 0.3, 
                          logfc.threshold = 0.6, test.use = "wilcox") %>% 
           group_by(cluster) %>% 
           arrange(desc(avg_log2FC), .by_group = TRUE)

# 创建交互式表格 (datatable)
markers_dt <- datatable(as.data.frame(markers),
                        extensions = c('KeyTable', 'Buttons', 'FixedColumns'),
                        class = 'cell-border stripe',
                        filter = 'top',
                        options = list(
                          dom = 'lrtipB',
                          buttons = list(list(
                            extend = 'collection',
                            buttons = c('csv', 'excel', 'pdf', 'copy'),
                            text = 'Export'
                          )),
                          pageLength = 20,
                          rownames = FALSE
                        ))

# 导出 HTML (使用DT的内置函数)
DT::saveWidget(markers_dt, 
               file = paste0(work_dir, "analysis_results/10.marker.dt.html"),
               selfcontained = TRUE)


In [ ]:
# # 同心毛细胞：PKD1、PKD2
# VlnPlot(handle_object, features = c('XLOC-004365#PKD1-MOUSE#O08852'), slot = "counts", log = TRUE)

# # 消化腺细胞：ASCL1、ASCL5
# VlnPlot(handle_object, features = c('XLOC-016568#ASCL1-XENLA#Q06234', 'XLOC-018098#ASCL5-MOUSE#M0QWB7'), slot = "counts", log = TRUE)

# # ● 刺丝胞（Nematocytes）：标记基因包括 minicollagen、toxin Crtx-A、NAS4、VMO1。
# VlnPlot(handle_object, features = c('XLOC-001212#NAS4-CAEEL#P55112', 'XLOC-000295#NAS4-CAEEL#P55112', 'XLOC-000900#NAS4-CAEEL#P55112', 'XLOC-011781#NAS4-CAEEL#P55112', 'XLOC-011858#NAS4-CAEEL#P55112', 'gene-evm.model.ptg000063l.1#NAS4-CAEEL#P55112', 'XLOC-015847#NAS4-CAEEL#P55112', 'XLOC-021316#NAS4-CAEEL#P55112', 'XLOC-021317#NAS4-CAEEL#P55112', 'XLOC-022161#NAS4-CAEEL#P55112'), slot = "counts", log = TRUE)
# VlnPlot(handle_object, features = c('XLOC-003934#VMO1-HUMAN#Q7Z5L0', 'XLOC-008936#VMO1-CHICK#P41366', 'XLOC-008935#VMO1-HUMAN#Q7Z5L0', 'XLOC-026768#VMO1-HUMAN#Q7Z5L0'), slot = "counts", log = TRUE)

# # ● 神经细胞（Neural Cells）：标记基因包括 ELAVL2、TUBA1A 和 SYT14。
# VlnPlot(handle_object, features = c('XLOC-002969#ELAV1-XENTR#Q6GLB5', 'gene-evm.model.ptg000031l.486#ELAV2-XENTR#Q28GD4'), slot = "counts", log = TRUE)
# VlnPlot(handle_object, features = c('XLOC-019965#SYT14-HUMAN#Q8NB59'), slot = "counts", log = TRUE)

# # ● 表皮/肌肉细胞（Epidermal/Muscle Cells）：标记基因包括 MLCK、MYS 和 POT1。
# VlnPlot(handle_object, features = c('XLOC-000820#MYLK-MOUSE#Q6PDN3', 'gene-evm.model.ptg000012l.823#MYLK-BOVIN#Q28824', 'XLOC-006281#MYLK-BOVIN#Q28824', 'XLOC-009507#MYLK-BOVIN#Q28824', 'XLOC-018994#MYLK-MOUSE#Q6PDN3', 'XLOC-019397#MYLK-MOUSE#Q6PDN3'), slot = "counts", log = TRUE)
# VlnPlot(handle_object, features = c('XLOC-004696#MYS-ARGIR#P24733'), slot = "counts", log = TRUE)

# # ● 胃皮层细胞（Gastrodermal Cells）：标记基因包括 CATL、Astacin-3 和 APLP。
# VlnPlot(handle_object, features = c('gene-evm.model.ptg000009l.1097#CATL-AEDAE#A0A1S4F2V5', 'XLOC-006269#CATL1-DROME#Q95029'), slot = "counts", log = TRUE)
# VlnPlot(handle_object, features = c('XLOC-003006#APLP-LOCMI#Q9U943', 'XLOC-003850#APLP-LOCMI#Q9U943'), slot = "counts", log = TRUE)

# # ● 腺体细胞（Gland Cells）：标记基因包括 CTRB2、CTR2、OVCH1 和 PRSS1。
# VlnPlot(handle_object, features = c('XLOC-002769#CTRB2-HUMAN#Q6GPI1'), slot = "counts", log = TRUE)
# VlnPlot(handle_object, features = c('gene-evm.model.ptg000015l.30#CTR2-CANLF#P04813', 'XLOC-011043#CTR2-XENLA#Q6DCE8', 'XLOC-026149#CTR2-XENLA#Q6DCE8'), slot = "counts", log = TRUE)
     

In [ ]:
grep("CTRB2", common_genes, value = TRUE, ignore.case = TRUE)

In [ ]:
# 保存 Seurat 对象为 .rds 文件
saveRDS(handle_object, file = paste0(work_dir, "analysis_results/Auco.seurat.rds"))

### 参考 Markers 进行细胞类型注释

In [ ]:
# 同心毛细胞：PKD1
VlnPlot(handle_object, features = c('XLOC-004365#PKD1-MOUSE#O08852'), slot = "counts", log = TRUE)

In [ ]:
# 消化腺细胞：ASCL1、ASCL5
VlnPlot(handle_object, features = c('XLOC-016568#ASCL1-XENLA#Q06234', 'XLOC-018098#ASCL5-MOUSE#M0QWB7'), slot = "counts", log = TRUE)

In [ ]:
# ● 刺丝胞（Nematocytes）：标记基因包括 minicollagen、toxin Crtx-A、NAS4、VMO1。
VlnPlot(handle_object, features = c('XLOC-001212#NAS4-CAEEL#P55112', 'XLOC-000295#NAS4-CAEEL#P55112', 'XLOC-000900#NAS4-CAEEL#P55112', 'XLOC-011781#NAS4-CAEEL#P55112', 'XLOC-011858#NAS4-CAEEL#P55112', 'gene-evm.model.ptg000063l.1#NAS4-CAEEL#P55112', 'XLOC-015847#NAS4-CAEEL#P55112', 'XLOC-021316#NAS4-CAEEL#P55112', 'XLOC-021317#NAS4-CAEEL#P55112', 'XLOC-022161#NAS4-CAEEL#P55112'), slot = "counts", log = TRUE)
VlnPlot(handle_object, features = c('XLOC-003934#VMO1-HUMAN#Q7Z5L0', 'XLOC-008936#VMO1-CHICK#P41366', 'XLOC-008935#VMO1-HUMAN#Q7Z5L0', 'XLOC-026768#VMO1-HUMAN#Q7Z5L0'), slot = "counts", log = TRUE)

In [ ]:
# ● 神经细胞（Neural Cells）：标记基因包括 ELAVL2、TUBA1A 和 SYT14。
VlnPlot(handle_object, features = c('XLOC-002969#ELAV1-XENTR#Q6GLB5', 'gene-evm.model.ptg000031l.486#ELAV2-XENTR#Q28GD4'), slot = "counts", log = TRUE)
VlnPlot(handle_object, features = c('XLOC-019965#SYT14-HUMAN#Q8NB59'), slot = "counts", log = TRUE)

In [ ]:
# ● 表皮/肌肉细胞（Epidermal/Muscle Cells）：标记基因包括 MLCK、MYS 和 POT1。
VlnPlot(handle_object, features = c('XLOC-000820#MYLK-MOUSE#Q6PDN3', 'gene-evm.model.ptg000012l.823#MYLK-BOVIN#Q28824', 'XLOC-006281#MYLK-BOVIN#Q28824', 'XLOC-009507#MYLK-BOVIN#Q28824', 'XLOC-018994#MYLK-MOUSE#Q6PDN3', 'XLOC-019397#MYLK-MOUSE#Q6PDN3'), slot = "counts", log = TRUE)
VlnPlot(handle_object, features = c('XLOC-004696#MYS-ARGIR#P24733'), slot = "counts", log = TRUE)

In [ ]:
# ● 胃皮层细胞（Gastrodermal Cells）：标记基因包括 CATL、Astacin-3 和 APLP。
VlnPlot(handle_object, features = c('gene-evm.model.ptg000009l.1097#CATL-AEDAE#A0A1S4F2V5', 'XLOC-006269#CATL1-DROME#Q95029'), slot = "counts", log = TRUE)
VlnPlot(handle_object, features = c('XLOC-003006#APLP-LOCMI#Q9U943', 'XLOC-003850#APLP-LOCMI#Q9U943'), slot = "counts", log = TRUE)

In [ ]:
# ● 腺体细胞（Gland Cells）：标记基因包括 CTRB2、CTR2、OVCH1 和 PRSS1。
VlnPlot(handle_object, features = c('XLOC-002769#CTRB2-HUMAN#Q6GPI1'), slot = "counts", log = TRUE)
VlnPlot(handle_object, features = c('gene-evm.model.ptg000015l.30#CTR2-CANLF#P04813', 'XLOC-011043#CTR2-XENLA#Q6DCE8', 'XLOC-026149#CTR2-XENLA#Q6DCE8'), slot = "counts", log = TRUE)

### 参考 Markers 进行细胞类型注释（第二轮）

In [ ]:
# 刺细胞 'XLOC-000295#NAS4-CAEEL#P55112', 'XLOC-022161#NAS4-CAEEL#P55112'
VlnPlot(handle_object, features = c('XLOC-000295#NAS4-CAEEL#P55112', 'XLOC-022161#NAS4-CAEEL#P55112'), slot = "counts", log = TRUE)

In [ ]:
# 神经细胞 'XLOC-019965#SYT14-HUMAN#Q8NB59'
VlnPlot(handle_object, features = c('XLOC-019965#SYT14-HUMAN#Q8NB59'), slot = "counts", log = TRUE)

In [ ]:
# 肌肉细胞 'XLOC-009507#MYLK-BOVIN#Q28824', 'XLOC-004696#MYS-ARGIR#P24733'
VlnPlot(handle_object, features = c('XLOC-009507#MYLK-BOVIN#Q28824', 'XLOC-004696#MYS-ARGIR#P24733'), slot = "counts", log = TRUE)

In [ ]:
# 胃皮层细胞 'gene-evm.model.ptg000009l.1097#CATL-AEDAE#A0A1S4F2V5', 'XLOC-006269#CATL1-DROME#Q95029', 'XLOC-003006#APLP-LOCMI#Q9U943'
VlnPlot(handle_object, features = c('gene-evm.model.ptg000009l.1097#CATL-AEDAE#A0A1S4F2V5', 'XLOC-006269#CATL1-DROME#Q95029', 'XLOC-003006#APLP-LOCMI#Q9U943'), slot = "counts", log = TRUE)

In [ ]:
# 腺体细胞 'XLOC-002769#CTRB2-HUMAN#Q6GPI1'
VlnPlot(handle_object, features = c('XLOC-002769#CTRB2-HUMAN#Q6GPI1'), slot = "counts", log = TRUE)

## 参考 Markers 进行细胞类型注释（综合多篇文献）

In [ ]:
# 表皮肌肉细胞（Epidermal/muscle cell，EM）
# MIc-c, CaM
# COL1A1, FBN2, SLC22, SVIL, MYH10
# MYL6B, CaM, SVIL, TPM1, TPM2
# MLCK, MYS, POT1
# TUBA1C, LMNA, MLCK
# RSPO2, FKBP2, HMCN1
# OBSCN, MYS, MyHc-2b
# POT1, FOXA2, MUC5AC
# GFP-like
# Clytia-specific 1
# DD3-3-like, Transgelin-like
# Tropomyosin-A
# ZP-containing-1
# Tropomyosin-C, ZP-containing-3
# Receptor kinase-like, Myosin light chain

# 表皮细胞（Epidermal cell，EM.ep）
# AAUR2.49356
# AaMELC12-like（melc12、MLE_BRAFL）
# MLC2-like-4（MLC2_DROME）
# F16CA-like-2（F16CA_XENLA）
# HMU-like-3（HMU_HALWD）
# DD3-like-1（DD3_DICDI）
# AAUR2.10977
# H1-like-1（HRH1_BOVIN）

# 条纹肌细胞（striated muscle cell，EM.st）
# AastMHC5（stmhc5、MYH4_CANLF）
# AAUR2.16461
# CO2A1-like-1（CO2A1_RAT）
# MLE1-like-2（MLE1_CHERA）
# MLE-like-1（MLE_BRAFL）
# PRC1-like-1（PRC1_MOUSE）
# SMTNL1, MYL6, PFN, RIM2, CHRNN

# 胃皮层细胞（Gastrodermal cell，GA）
# apoLp
# apoLp, Serpin, COL1A1
# CATL, Astacin-3, APLP
# APLP, CATL, Astacin-3
# CYT, TEFF1, Galaxin
# mesoglein, SPON1, SC6A1
# ITLN1, ITLN2, IRF2
# SAV2-like-1（SAV2_STRVL）
# AAUR2.25075
# GGH-like-2（GGH_HUMAN）
# AAUR2.47263
# AaAPLP-like2（aplp、APLP_LOCMI）
# CSMD1-like-1（CSMD1_MOUSE）
# IFI30-a, 
# IFI30-b, CathepsinL
# Innexin-like, PIGT
# APOB, Vitellogenin-like
# PAMR1,
# Antimicrobial-like
# VMO1
# Fibrillar-collagen-B/C

# 神经细胞（Neural cell，NE）
# SYT16
# TUBA1A, SYT15, NCAM, ELAVL1
# TUBA, OCLC, AVIL, VGSC
# ELAVL2, TUBA1A, SYT14
# TUBA1A, ELAVL2, SYT14
# CAB32, MATN3, NELL1, GPN2
# AAUR2.23891
# AAUR2.5836
# LWA-like-1（LWA_ANTEL）
# MLRP2-like-7（MLRP2_ACRMI）
# ACH1-like-3（ACH1_CAEBR）
# EMX1-like-3（EMX1_XENTR）
# AAUR2.10977
# HE12-like-3（HE12_DANRE）
# AAUR2.38264
# AAUR2.50651
# TMPS6-like-3（TMPS6_MOUSE）
# AAUR2.20890
# CACNA1E, TRPC4（调控肌肉收缩）
# AVIL, HMCN1（感知功能）
# APBA2, PARD3（突触可塑性）
# ARV1, Clytia-specific 2
# PP5 Rfamide
# Tubulin-A
# Synaptotagmin
# VGSC
# SOX14, Lmx1A
# PRFA-like-1（PRFA_POLPE）
# CUBN-like-38（CUBN_RAT）
# MIB1-like-3（MIB1_MOUSE）
# NRX3A-like-2（NRX3A_HUMAN）
# MFGM-like-21（MFGM_PIG）
# GNAI1-like-1（GNAI1_CHICK）
# DAG1-like-6（DAG1_BOVIN）
# PRFA-like-1（PRFA_POLPE）
# VIAAT-like-8（VIAAT_XENTR）
# BCAL2-like-3（BCAL2_ARATH）
# AAUR2.48906
# RS2-like-1（GORS2_HUMAN）

# 运动神经元（Swim motor neurons，NE.sw）
# DCTN1、NRP1A、KIF13B

# 刺细胞（Cnidocytes/nematocyte cells，CN）
# Minicollagen, Nematogalectin
# Minicollagen, ShKT, Toxin TX2, Minicollagen, DKK3
# Nematogalectin×3, Minicollagen
# ShKT, Nematogalectin×3
# minicollagen, toxin Crtx-A, NAS4, VMO1
# minicollagen, NST1, FBN1
# NAS4, toxin Crtx-A, VMO1
# AaNemGal-like（nemgal-like、SML_SCONI）、AaTBA1C-like1（tba1c、TBA1C_MOUSE）
# MYC-like-7（myc4、MYC_PONPY）
# PK1L3-like-3（PK1L3_MOUSE）
# FAZ1-like-3
# NAS15-like-18（NAS15_CAEEL）
# FUCL6-like-3（FUCL6_ANGJA）
# AaTBA1C-like1（tba1c、TBA1C_MOUSE）
# PK1L2-like-17（PK1L2_MOUSE）
# NETR-like-1（NETR_PANTR）
# AURE-like-1（AURE_AURAU）
# CPD, PHF24
# EPRS
# QTRT1, Collagen-like
# Thioredoxin-containing
# Profilin-containing
# Nematocilin, CDC23
# CEP135-like

# 腺体细胞（Gland cell，GL）
# CTRB2, CTR2, OVCH1, PRSS1
# CTRB2, CTR2, NAS15
# SVEP1, PLA2, ENDOU
# PRSS1, OVCH1, PRSS27
# Gal_Lectin-containing
# NIT2
# Astacin-1, Astacin-2
# Trypsin-like 1
# Trypsin-like 2

# 分泌腺细胞（Secretory gland cell，GL.se）
# MUC2, CREB3L
# FKBP13, MUC2, P4HTM, CREB3L

# 黏液腺细胞（Mucin gland cell，GL.mu）
# AGRL3-like-2（AGRL3_BOVIN）
# MATN3-like-1（MATN3_CHICK）
# CADN-like-13（CADN_ACRMI）
# CADN-like-2（CADN_ACRMI）
# MLRP1-like-9（MLRP1_ACRMI）
# AAUR2.16461
# AAUR2.9270

# 消化腺细胞（Digestive gland cell，GL.di）
# SVEP1-like-3（SVEP1_MOUSE）
# AAUR2.21790
# CEL2A-like-9（CEL2A_PIG）
# CEL2A-like-8（CEL2A_RAT）
# AaTrypsin2（trypsin2、CTRB2_HUMAN）
# CTX1-like-3（CTX1_CARRA）
# SVEP1-like-10（SVEP1_MOUSE）
# LIPR2-like-7（LIPR2_MYOCO）
# NAS4-like-14（NAS4_CAEEL）
# ASCL1、ASCL5

# 干细胞/生殖细胞（Stem/germline cell，SG）
# SMC2, SMC4
# CENPA, TOP2A, SMC4
# ZP-containing-2
# FMN-reductiase
# DDX39A/B, PA2G4
# ADAMTS17/19, SKP1

# 感觉细胞/同心毛细胞（Hair cell，HA）
# PKD1、PKD2
# CFAP141（T. rubra无）、TRPC4、CACNA1E、LOXHD1、KIF13B、DNAH1
# 平衡囊形成相关：LRIG3、NOTUM2
# 纤毛发生相关：KIF14
# 纤毛运动相关：NPC1、FAM166B、DNAHs家族
# 感觉毛细胞（hair cell）功能相关：LOXHD1、LRP5

In [ ]:
library(Seurat)
library(patchwork)

In [ ]:
length(common_genes)

### 表皮肌肉细胞（Epidermal/muscle cell，EM）

In [ ]:
query_list <- c(
  "CALM",
  "CO1A1",
  "FBN2",
  "SLC22",
  "SVIL",
  "MYH10",
  "MYL6B",
  "TPM1",
  "TPM2",
  "MLCK3",
  "ITBX",
  "POT1",
  "TBA1C",
  "LMNA",
  "RSPO2",
  "FKBP2",
  "HMCN1",
  "OBSCN",
  "MYH4",
  "FOXA2",
  "MUC5A",
  "GFP",
  "DD3",
  "TAGLN",
  "ZP1",
  "TPM3",
  "ZP3",
  "XA21",
  "MYL"
)

res_list <- c()

# 1. 逐个打印并累加hits
for (q in query_list) {
  hits <- grep(q, common_genes, value = TRUE)
  cat(q, ":", paste(hits, collapse = ","), "\n")
  if (length(hits) > 0) {
    res_list <- c(res_list, hits)
  }
}

print(res_list)

# 2. 每行展示3个基因的小提琴图
if (length(res_list) > 0) {
  # 设置Jupyter的绘图画布大小，适合大图显示
  options(repr.plot.width = 15, repr.plot.height = 4 * ceiling(length(res_list)/3))

  n <- 3 # 每行最多3个
  plots <- lapply(res_list, function(gene) {
    VlnPlot(handle_object, features = gene, pt.size = 0.1) +
      ggtitle(gene) +
      theme(plot.title = element_text(size = 12))
  })

  combined_plot <- wrap_plots(plots, ncol = n)
  print(combined_plot)
}

#### 表皮细胞（Epidermal cell，EM.ep）

In [ ]:
query_list <- c("MLE","MLC2","F16CA","#HUM","DD3","HRH1")

res_list <- c()

# 1. 逐个打印并累加hits
for (q in query_list) {
  hits <- grep(q, common_genes, value = TRUE)
  cat(q, ":", paste(hits, collapse = ","), "\n")
  if (length(hits) > 0) {
    res_list <- c(res_list, hits)
  }
}

print(res_list)

# 2. 每行展示3个基因的小提琴图
if (length(res_list) > 0) {
  # 设置Jupyter的绘图画布大小，适合大图显示
  options(repr.plot.width = 15, repr.plot.height = 4 * ceiling(length(res_list)/3))

  n <- 3 # 每行最多3个
  plots <- lapply(res_list, function(gene) {
    VlnPlot(handle_object, features = gene, pt.size = 0.1) +
      ggtitle(gene) +
      theme(plot.title = element_text(size = 12))
  })

  combined_plot <- wrap_plots(plots, ncol = n)
  print(combined_plot)
}

#### 条纹肌细胞（striated muscle cell，EM.st）

In [ ]:
query_list <- c(
  "MYH4",
  "CO2A1",
  "MLE1",
  "MLE",
  "PRC1",
  "SMTL1",
  "MYL6",
  "PROF",
  "RIM2",
  "A0A6J8ADN4"
)

res_list <- c()

# 1. 逐个打印并累加hits
for (q in query_list) {
  hits <- grep(q, common_genes, value = TRUE)
  cat(q, ":", paste(hits, collapse = ","), "\n")
  if (length(hits) > 0) {
    res_list <- c(res_list, hits)
  }
}

print(res_list)

# 2. 每行展示3个基因的小提琴图
if (length(res_list) > 0) {
  # 设置Jupyter的绘图画布大小，适合大图显示
  options(repr.plot.width = 15, repr.plot.height = 4 * ceiling(length(res_list)/3))

  n <- 3 # 每行最多3个
  plots <- lapply(res_list, function(gene) {
    VlnPlot(handle_object, features = gene, pt.size = 0.1) +
      ggtitle(gene) +
      theme(plot.title = element_text(size = 12))
  })

  combined_plot <- wrap_plots(plots, ncol = n)
  print(combined_plot)
}

### 胃皮层细胞（Gastrodermal cell，GA）

In [ ]:
query_list <- c(
  "APLP",
  "SPB12",
  "CO1A1",
  "CATL1",
  "NAS5",
  "APLP1",
  "CYT19",
  "TEFF1",
  "GXN",
  "A0A097J9X4",
  "SPON1",
  "SC6A1",
  "ITLN1",
  "ITLN2",
  "IRF2",
  "SAV2",
  "GGH",
  "APLP",
  "CSMD1",
  "GILT",
  "GILT",
  "A0A646QFB6",
  "INX3",
  "PIGT",
  "APOB",
  "VIT",
  "PAMR1",
  "AMP",
  "VMO1"
)

res_list <- c()

# 1. 逐个打印并累加hits
for (q in query_list) {
  hits <- grep(q, common_genes, value = TRUE)
  cat(q, ":", paste(hits, collapse = ","), "\n")
  if (length(hits) > 0) {
    res_list <- c(res_list, hits)
  }
}

print(res_list)

# 2. 每行展示3个基因的小提琴图
if (length(res_list) > 0) {
  # 设置Jupyter的绘图画布大小，适合大图显示
  options(repr.plot.width = 15, repr.plot.height = 4 * ceiling(length(res_list)/3))

  n <- 3 # 每行最多3个
  plots <- lapply(res_list, function(gene) {
    VlnPlot(handle_object, features = gene, pt.size = 0.1) +
      ggtitle(gene) +
      theme(plot.title = element_text(size = 12))
  })

  combined_plot <- wrap_plots(plots, ncol = n)
  print(combined_plot)
}

### 神经细胞（Neural cell，NE）

In [ ]:
query_list <- c(
  "SYT16",
  "TBA1A",
  "SYT15",
  "NCAM1",
  "ELAV1",
  "DNMBP",
  "AVIL",
  "TXH3",
  "ELAV2",
  "SYT14",
  "CAB32",
  "MATN3",
  "NELL1",
  "GPN2",
  "LWA",
  "MLRP2",
  "ACH1",
  "EMX1",
  "HE12",
  "TMPS6",
  "CAC1E",
  "TRPC4",
  "HMCN1",
  "APBA2",
  "PARD3",
  "ARV1",
  "TBCA",
  "SYT14",
  "SOX14",
  "LMX1A",
  "PRFA",
  "CUBN",
  "MIB1",
  "NRX3A",
  "MFGM",
  "GNAI1",
  "DAG1",
  "VIAAT",
  "BCAL2",
  "GORS2"
)

res_list <- c()

# 1. 逐个打印并累加hits
for (q in query_list) {
  hits <- grep(q, common_genes, value = TRUE)
  cat(q, ":", paste(hits, collapse = ","), "\n")
  if (length(hits) > 0) {
    res_list <- c(res_list, hits)
  }
}

print(res_list)

# 2. 每行展示3个基因的小提琴图
if (length(res_list) > 0) {
  # 设置Jupyter的绘图画布大小，适合大图显示
  options(repr.plot.width = 15, repr.plot.height = 4 * ceiling(length(res_list)/3))

  n <- 3 # 每行最多3个
  plots <- lapply(res_list, function(gene) {
    VlnPlot(handle_object, features = gene, pt.size = 0.1) +
      ggtitle(gene) +
      theme(plot.title = element_text(size = 12))
  })

  combined_plot <- wrap_plots(plots, ncol = n)
  print(combined_plot)
}

#### 运动神经元（Swim motor neurons，NE.sw）

In [ ]:
query_list <- c(
  "DCTN1",
  "NRP1A",
  "KI13B"
)

res_list <- c()

# 1. 逐个打印并累加hits
for (q in query_list) {
  hits <- grep(q, common_genes, value = TRUE)
  cat(q, ":", paste(hits, collapse = ","), "\n")
  if (length(hits) > 0) {
    res_list <- c(res_list, hits)
  }
}

print(res_list)

# 2. 每行展示3个基因的小提琴图
if (length(res_list) > 0) {
  # 设置Jupyter的绘图画布大小，适合大图显示
  options(repr.plot.width = 15, repr.plot.height = 4 * ceiling(length(res_list)/3))

  n <- 3 # 每行最多3个
  plots <- lapply(res_list, function(gene) {
    VlnPlot(handle_object, features = gene, pt.size = 0.1) +
      ggtitle(gene) +
      theme(plot.title = element_text(size = 12))
  })

  combined_plot <- wrap_plots(plots, ncol = n)
  print(combined_plot)
}

### 刺细胞（Cnidocytes/nematocyte cells，CN）

In [ ]:
query_list <- c(
  "A0A7M5X0Z9",
  "E0D8Q4",
  "SKPO1",
  "#TX2",
  "DKK3",
  "A0A7M5X0Z9",
  "JTX21",
  "NAS4",
  "VMO1",
  "NST1",
  "FBN1",
  "SML",
  "TBA1C",
  "#MYC",
  "PK1L3",
  "FAZ1",
  "NAS15",
  "FUCL6",
  "PK1L2",
  "NETR",
  "AURE",
  "#CPD",
  "PHF24",
  "SYEP",
  "QTRT2",
  "CO7A1",
  "TXND2",
  "PROF",
  "B3A0S9",
  "CDC23",
  "CP135"
)

res_list <- c()

# 1. 逐个打印并累加hits
for (q in query_list) {
  hits <- grep(q, common_genes, value = TRUE)
  cat(q, ":", paste(hits, collapse = ","), "\n")
  if (length(hits) > 0) {
    res_list <- c(res_list, hits)
  }
}

print(res_list)

# 2. 每行展示3个基因的小提琴图
if (length(res_list) > 0) {
  # 设置Jupyter的绘图画布大小，适合大图显示
  options(repr.plot.width = 15, repr.plot.height = 4 * ceiling(length(res_list)/3))

  n <- 3 # 每行最多3个
  plots <- lapply(res_list, function(gene) {
    VlnPlot(handle_object, features = gene, pt.size = 0.1) +
      ggtitle(gene) +
      theme(plot.title = element_text(size = 12))
  })

  combined_plot <- wrap_plots(plots, ncol = n)
  print(combined_plot)
}

### 腺体细胞（Gland cell，GL）

In [ ]:
query_list <- c(
  "CTRB2",
  "CTR2",
  "OVCH1",
  "TRY1",
  "NAS15",
  "SVEP1",
  "PLA2",
  "ENDOU",
  "PRS27",
  "NIT2",
  "ASTL",
  "ASTL",
  "TRY1",
  "TRY1"
)

res_list <- c()

# 1. 逐个打印并累加hits
for (q in query_list) {
  hits <- grep(q, common_genes, value = TRUE)
  cat(q, ":", paste(hits, collapse = ","), "\n")
  if (length(hits) > 0) {
    res_list <- c(res_list, hits)
  }
}

print(res_list)

# 2. 每行展示3个基因的小提琴图
if (length(res_list) > 0) {
  # 设置Jupyter的绘图画布大小，适合大图显示
  options(repr.plot.width = 15, repr.plot.height = 4 * ceiling(length(res_list)/3))

  n <- 3 # 每行最多3个
  plots <- lapply(res_list, function(gene) {
    VlnPlot(handle_object, features = gene, pt.size = 0.1) +
      ggtitle(gene) +
      theme(plot.title = element_text(size = 12))
  })

  combined_plot <- wrap_plots(plots, ncol = n)
  print(combined_plot)
}

#### 分泌腺细胞（Secretory gland cell，GL.se）

In [ ]:
query_list <- c(
  "MUC2",
  "FKB13",
  "MUC2",
  "P4HTM"
)

res_list <- c()

# 1. 逐个打印并累加hits
for (q in query_list) {
  hits <- grep(q, common_genes, value = TRUE)
  cat(q, ":", paste(hits, collapse = ","), "\n")
  if (length(hits) > 0) {
    res_list <- c(res_list, hits)
  }
}

print(res_list)

# 2. 每行展示3个基因的小提琴图
if (length(res_list) > 0) {
  # 设置Jupyter的绘图画布大小，适合大图显示
  options(repr.plot.width = 15, repr.plot.height = 4 * ceiling(length(res_list)/3))

  n <- 3 # 每行最多3个
  plots <- lapply(res_list, function(gene) {
    VlnPlot(handle_object, features = gene, pt.size = 0.1) +
      ggtitle(gene) +
      theme(plot.title = element_text(size = 12))
  })

  combined_plot <- wrap_plots(plots, ncol = n)
  print(combined_plot)
}

#### 黏液腺细胞（Mucin gland cell，GL.mu）

In [ ]:
query_list <- c(
  "AGRL3",
  "MATN3",
  "CADN",
  "CADN",
  "MLRP1"
)

res_list <- c()

# 1. 逐个打印并累加hits
for (q in query_list) {
  hits <- grep(q, common_genes, value = TRUE)
  cat(q, ":", paste(hits, collapse = ","), "\n")
  if (length(hits) > 0) {
    res_list <- c(res_list, hits)
  }
}

print(res_list)

# 2. 每行展示3个基因的小提琴图
if (length(res_list) > 0) {
  # 设置Jupyter的绘图画布大小，适合大图显示
  options(repr.plot.width = 15, repr.plot.height = 4 * ceiling(length(res_list)/3))

  n <- 3 # 每行最多3个
  plots <- lapply(res_list, function(gene) {
    VlnPlot(handle_object, features = gene, pt.size = 0.1) +
      ggtitle(gene) +
      theme(plot.title = element_text(size = 12))
  })

  combined_plot <- wrap_plots(plots, ncol = n)
  print(combined_plot)
}

#### 消化腺细胞（Digestive gland cell，GL.di）

In [ ]:
query_list <- c(
  "SVEP1",
  "CEL2A",
  "CEL2A",
  "CTRB2",
  "CTX1",
  "SVEP1",
  "LIPR2",
  "NAS4",
  "ASCL1",
  "ASCL5"
)

res_list <- c()

# 1. 逐个打印并累加hits
for (q in query_list) {
  hits <- grep(q, common_genes, value = TRUE)
  cat(q, ":", paste(hits, collapse = ","), "\n")
  if (length(hits) > 0) {
    res_list <- c(res_list, hits)
  }
}

print(res_list)

# 2. 每行展示3个基因的小提琴图
if (length(res_list) > 0) {
  # 设置Jupyter的绘图画布大小，适合大图显示
  options(repr.plot.width = 15, repr.plot.height = 4 * ceiling(length(res_list)/3))

  n <- 3 # 每行最多3个
  plots <- lapply(res_list, function(gene) {
    VlnPlot(handle_object, features = gene, pt.size = 0.1) +
      ggtitle(gene) +
      theme(plot.title = element_text(size = 12))
  })

  combined_plot <- wrap_plots(plots, ncol = n)
  print(combined_plot)
}

### 干细胞/生殖细胞（Stem/germline cell，SG）

In [ ]:
query_list <- c(
  "SMC2",
  "SMC4",
  "CENPA",
  "TOP2A",
  "#ZP2",
  "FMN1",
  "DX39A",
  "DX39B",
  "PA2G4",
  "ATS17",
  "ATS19",
  "SKP1"
)

res_list <- c()

# 1. 逐个打印并累加hits
for (q in query_list) {
  hits <- grep(q, common_genes, value = TRUE)
  cat(q, ":", paste(hits, collapse = ","), "\n")
  if (length(hits) > 0) {
    res_list <- c(res_list, hits)
  }
}

print(res_list)

# 2. 每行展示3个基因的小提琴图
if (length(res_list) > 0) {
  # 设置Jupyter的绘图画布大小，适合大图显示
  options(repr.plot.width = 15, repr.plot.height = 4 * ceiling(length(res_list)/3))

  n <- 3 # 每行最多3个
  plots <- lapply(res_list, function(gene) {
    VlnPlot(handle_object, features = gene, pt.size = 0.1) +
      ggtitle(gene) +
      theme(plot.title = element_text(size = 12))
  })

  combined_plot <- wrap_plots(plots, ncol = n)
  print(combined_plot)
}

### 感觉细胞（Hair cell，HA）

In [ ]:
query_list <- c(
  "PKD1",
  "PKD2",
  "CP141",
  "TRPC4",
  "CAC1E",
  "LOXH1",
  "KI13B",
  "DYH1",
  "LRIG3",
  "NOTU2",
  "KIF14",
  "NPC1",
  "CMI2B",
  "DYH1",
  "LRP5",
  "MYO7A",
  "LOXH1",
  "PO4F3",
  "OTOF",
  "TBA1A",
  "AVIL"
)

res_list <- c()

# 1. 逐个打印并累加hits
for (q in query_list) {
  hits <- grep(q, common_genes, value = TRUE)
  cat(q, ":", paste(hits, collapse = ","), "\n")
  if (length(hits) > 0) {
    res_list <- c(res_list, hits)
  }
}

print(res_list)

# 2. 每行展示3个基因的小提琴图
if (length(res_list) > 0) {
  # 设置Jupyter的绘图画布大小，适合大图显示
  options(repr.plot.width = 15, repr.plot.height = 4 * ceiling(length(res_list)/3))

  n <- 3 # 每行最多3个
  plots <- lapply(res_list, function(gene) {
    VlnPlot(handle_object, features = gene, pt.size = 0.1) +
      ggtitle(gene) +
      theme(plot.title = element_text(size = 12))
  })

  combined_plot <- wrap_plots(plots, ncol = n)
  print(combined_plot)
}

## 后续分析

In [ ]:
library(Seurat)
library(patchwork)

In [ ]:
length(common_genes)

### 绘制Makers的小提琴图

In [ ]:
# 表皮肌肉细胞Markers
res_list <- c(
  "XLOC-011330#CALM-MACPY#Q40302",
  "XLOC-014937#SVIL-BOVIN#O46385",
  "XLOC-016636#MYL6B-HUMAN#P14649",
  "XLOC-018113#MYL6B-HUMAN#P14649",
  "XLOC-026007#TPM1-PODCA#P41114",
  "XLOC-024876#TPM2-PODCA#Q9U5M4",
  "XLOC-004173#DD3-DICDI#Q58A42",
  "gene-evm.model.ptg000006l.268#MYL1-DANRE#Q6P0G6",
  "XLOC-016635#MYL1-DANRE#Q6P0G6",
  "XLOC-026552#MLE-BRAFL#Q17133",
  "gene-evm.model.ptg000002l.132#MLE-BRAFL#Q17133"
)

if (length(res_list) > 0) {
  # 设置Jupyter的绘图画布大小，适合大图显示
  options(repr.plot.width = 15, repr.plot.height = 4 * ceiling(length(res_list)/3))

  n <- 3 # 每行最多3个
  plots <- lapply(res_list, function(gene) {
    VlnPlot(handle_object, features = gene, pt.size = 0.1) +
      ggtitle(gene) +
      theme(plot.title = element_text(size = 12))
  })

  combined_plot <- wrap_plots(plots, ncol = n)
  print(combined_plot)
}

In [ ]:
# 胃皮层细胞Markers
res_list <- c(
  "XLOC-003006#APLP-LOCMI#Q9U943",
  "XLOC-006269#CATL1-DROME#Q95029",
  "XLOC-009813#IRF2-CHICK#Q98925",
  "XLOC-002530#CSMD1-MOUSE#Q923L3"
)

if (length(res_list) > 0) {
  # 设置Jupyter的绘图画布大小，适合大图显示
  options(repr.plot.width = 15, repr.plot.height = 4 * ceiling(length(res_list)/3))

  n <- 3 # 每行最多3个
  plots <- lapply(res_list, function(gene) {
    VlnPlot(handle_object, features = gene, pt.size = 0.1) +
      ggtitle(gene) +
      theme(plot.title = element_text(size = 12))
  })

  combined_plot <- wrap_plots(plots, ncol = n)
  print(combined_plot)
}

In [ ]:
# 刺细胞Markers
res_list <- c(
  "XLOC-010665#MYC-MARMO#P22555"
)

if (length(res_list) > 0) {
  # 设置Jupyter的绘图画布大小，适合大图显示
  options(repr.plot.width = 15, repr.plot.height = 4 * ceiling(length(res_list)/3))

  n <- 3 # 每行最多3个
  plots <- lapply(res_list, function(gene) {
    VlnPlot(handle_object, features = gene, pt.size = 0.1) +
      ggtitle(gene) +
      theme(plot.title = element_text(size = 12))
  })

  combined_plot <- wrap_plots(plots, ncol = n)
  print(combined_plot)
}

In [ ]:
# 腺体细胞Markers
res_list <- c(
  "XLOC-002769#CTRB2-HUMAN#Q6GPI1",
  "gene-evm.model.ptg000015l.30#CTR2-CANLF#P04813",
  "XLOC-020391#TRY1-HUMAN#P07477",
  "XLOC-001150#SVEP1-RAT#P0C6B8",
  "XLOC-000359#SVEP1-MOUSE#A2AVA0",
  "XLOC-006327#CEL2A-RAT#P00774"
)

if (length(res_list) > 0) {
  # 设置Jupyter的绘图画布大小，适合大图显示
  options(repr.plot.width = 15, repr.plot.height = 4 * ceiling(length(res_list)/3))

  n <- 3 # 每行最多3个
  plots <- lapply(res_list, function(gene) {
    VlnPlot(handle_object, features = gene, pt.size = 0.1) +
      ggtitle(gene) +
      theme(plot.title = element_text(size = 12))
  })

  combined_plot <- wrap_plots(plots, ncol = n)
  print(combined_plot)
}

In [ ]:
# 干细胞/生殖细胞Markers
res_list <- c(
  "gene-evm.model.ptg000031l.94#SMC2-XENLA#P50533",
  "XLOC-003196#SMC2-XENLA#P50533",
  "XLOC-022183#SMC4-XENLA#P50532",
  "XLOC-009343#ATS17-HUMAN#Q8TE56"
)

if (length(res_list) > 0) {
  # 设置Jupyter的绘图画布大小，适合大图显示
  options(repr.plot.width = 15, repr.plot.height = 4 * ceiling(length(res_list)/3))

  n <- 3 # 每行最多3个
  plots <- lapply(res_list, function(gene) {
    VlnPlot(handle_object, features = gene, pt.size = 0.1) +
      ggtitle(gene) +
      theme(plot.title = element_text(size = 12))
  })

  combined_plot <- wrap_plots(plots, ncol = n)
  print(combined_plot)
}

In [ ]:
# 感觉细胞Markers
res_list <- c(
  "XLOC-001822#PKD2-BOVIN#Q4GZT3",
  "XLOC-015184#CAC1E-MOUSE#Q61290",
  "gene-evm.model.ptg000003l.332#TBA1A-CHICK#P02552",
  "XLOC-006274#AVIL-RAT#Q9WU06"
)

if (length(res_list) > 0) {
  # 设置Jupyter的绘图画布大小，适合大图显示
  options(repr.plot.width = 15, repr.plot.height = 4 * ceiling(length(res_list)/3))

  n <- 3 # 每行最多3个
  plots <- lapply(res_list, function(gene) {
    VlnPlot(handle_object, features = gene, pt.size = 0.1) +
      ggtitle(gene) +
      theme(plot.title = element_text(size = 12))
  })

  combined_plot <- wrap_plots(plots, ncol = n)
  print(combined_plot)
}

In [ ]:
# 神经细胞Markers
res_list <- c(
  "XLOC-019965#SYT14-HUMAN#Q8NB59",
  "gene-evm.model.ptg000013l.805#CAB32-DROME#P41044",
  "gene-evm.model.ptg000022l.585#ACH1-CAEEL#P48180",
  "gene-evm.model.ptg000001l.686#SOX14-DANRE#Q32PP9",
  "gene-evm.model.ptg000012l.280#GORS2-HUMAN#Q9H8Y8"
)

if (length(res_list) > 0) {
  # 设置Jupyter的绘图画布大小，适合大图显示
  options(repr.plot.width = 15, repr.plot.height = 4 * ceiling(length(res_list)/3))

  n <- 3 # 每行最多3个
  plots <- lapply(res_list, function(gene) {
    VlnPlot(handle_object, features = gene, pt.size = 0.1) +
      ggtitle(gene) +
      theme(plot.title = element_text(size = 12))
  })

  combined_plot <- wrap_plots(plots, ncol = n)
  print(combined_plot)
}

### DotPlot

In [ ]:
library(ggplot2)
# 提取每个聚类的前 10 个差异表达基因
top_genes <- markers %>% 
  group_by(cluster) %>% 
  top_n(n = 3, wt = avg_log2FC)

# 绘制 DotPlot
# DotPlot(handle_object, features = unique(top10_genes$gene), group.by = "seurat_clusters") + 
#   RotatedAxis() + 
#   theme(axis.text.x = element_text(angle = 45, hjust = 1, size = 8))


pDotPlot <- DotPlot(handle_object, features = unique(top_genes$gene), group.by = "seurat_clusters") + 
  scale_color_viridis_c() + 
  RotatedAxis() + 
  theme(axis.text.x = element_text(angle = 90, hjust = 1, size = 8))

pdf(paste0(work_dir,'analysis_results/13.gene.DotPlot.pdf'))
print(pDotPlot)
dev.off()
pDotPlot

In [ ]:
library(dplyr)

# 对 markers 进行分组，按基因筛选出高表达的泛基因
filtered_markers <- markers %>%
  group_by(gene) %>%
  filter(max(pct.1) < 0.8)  # 基因在任何群体中表达比例不能高于 90%

# 提取每个聚类的前 10 个差异表达基因
top_genes <- filtered_markers %>% 
  group_by(cluster) %>% 
  top_n(n = 3, wt = avg_log2FC)

pDotPlotFiltered <- DotPlot(handle_object, features = unique(top_genes$gene), group.by = "seurat_clusters") + 
  scale_color_viridis_c() + 
  RotatedAxis() + 
  theme(axis.text.x = element_text(angle = 90, hjust = 1, size = 8))

pdf(paste0(work_dir,'analysis_results/14.gene.DotPlotFiltered.pdf'))
print(pDotPlotFiltered)
dev.off()
pDotPlotFiltered

### 基因家族在单细胞数据中分析

In [ ]:
# gene-evm.model.ptg000012l.203#DUS10-MOUSE#Q9ESS0
# XLOC-002473#CAC1A-APIME#C9D7C2
# XLOC-005591#RBM3-MOUSE#O89086
# XLOC-007156#KALRN-MOUSE#A2CG49
# XLOC-013650#TBA1-PARLI#P18258
# XLOC-015712#NCAH-DROME#P42325
# XLOC-026592#DUS4-CHICK#Q9PW71

In [ ]:
# 显著扩张的神经细胞基因Markers分析
res_list <- c(
    'gene-evm.model.ptg000012l.203#DUS10-MOUSE#Q9ESS0', 
    'XLOC-022161#NAS4-CAEEL#P55112',
    'XLOC-002473#CAC1A-APIME#C9D7C2',
    'XLOC-005591#RBM3-MOUSE#O89086',
    'XLOC-007156#KALRN-MOUSE#A2CG49',
    'XLOC-013650#TBA1-PARLI#P18258',
    'XLOC-015712#NCAH-DROME#P42325',
    'XLOC-026592#DUS4-CHICK#Q9PW71'
)

if (length(res_list) > 0) {
  # 设置Jupyter的绘图画布大小，适合大图显示
  options(repr.plot.width = 15, repr.plot.height = 4 * ceiling(length(res_list)/3))

  n <- 3 # 每行最多3个
  plots <- lapply(res_list, function(gene) {
    VlnPlot(handle_object, features = gene, pt.size = 0.1) +
      ggtitle(gene) +
      theme(plot.title = element_text(size = 12))
  })

  combined_plot <- wrap_plots(plots, ncol = n)
  print(combined_plot)
}

In [ ]:
# 显著扩张的神经细胞基因Markers分析
res_list <- c(
    'XLOC-002473#CAC1A-APIME#C9D7C2',
    'XLOC-007156#KALRN-MOUSE#A2CG49',
    'XLOC-013650#TBA1-PARLI#P18258',
    'XLOC-026592#DUS4-CHICK#Q9PW71'
)

if (length(res_list) > 0) {
  # 设置Jupyter的绘图画布大小，适合大图显示
  options(repr.plot.width = 15, repr.plot.height = 4 * ceiling(length(res_list)/3))

  n <- 3 # 每行最多3个
  plots <- lapply(res_list, function(gene) {
    VlnPlot(handle_object, features = gene, pt.size = 0.1) +
      ggtitle(gene) +
      theme(plot.title = element_text(size = 12))
  })

  combined_plot <- wrap_plots(plots, ncol = n)
  print(combined_plot)
}